In [42]:
!pip -q install \
transformers \
accelerate \
sentence-transformers \
faiss-cpu \
bitsandbytes \
einops

In [41]:
!pip install -q pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 75.1 MB/s eta 0:00:00:00:0100:01


In [43]:
import fitz
import os

documents = []

for pdf in pdf_files:

    doc = fitz.open(pdf)

    filename = os.path.basename(pdf)

    print("Reading:", filename)

    for page_num in range(len(doc)):

        page = doc.load_page(page_num)

        text = page.get_text()

        if text.strip():

            documents.append({
                "source": filename,
                "page": page_num + 1,
                "text": text
            })

print("Total Pages:", len(documents))

Reading: SBI_Arbitrage_Opportunities_Fund.pdf
Reading: SBI_Corporate_Bond_Fund.pdf
Reading: SBI_Dividend_Yield_Fund.pdf
Reading: SBI_Fixed_Maturity_Plan_(FMP)-_Series_47_(1434_Days).pdf
Total Pages: 347


In [44]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [45]:
import os

pdf_folder = "/kaggle/input"

pdf_files = []

for root, dirs, files in os.walk(pdf_folder):

    for file in files:

        if file.endswith(".pdf"):

            pdf_files.append(
                os.path.join(root, file)
            )

print("PDF Files Found:")

for pdf in pdf_files:
    print(pdf)

PDF Files Found:
/kaggle/input/datasets/reneekasharma/sbi-chatbot-pdfs/SBI_Arbitrage_Opportunities_Fund.pdf
/kaggle/input/datasets/reneekasharma/sbi-chatbot-pdfs/SBI_Corporate_Bond_Fund.pdf
/kaggle/input/datasets/reneekasharma/sbi-chatbot-pdfs/SBI_Dividend_Yield_Fund.pdf
/kaggle/input/datasets/reneekasharma/sbi-chatbot-pdfs/SBI_Fixed_Maturity_Plan_(FMP)-_Series_47_(1434_Days).pdf


In [46]:
documents = []

for pdf in pdf_files:

    reader = PdfReader(pdf)

    filename = os.path.basename(pdf)

    print("Reading:", filename)

    for page_number, page in enumerate(reader.pages):

        text = page.extract_text()

        if text:

            documents.append({

                "source": filename,

                "page": page_number + 1,

                "text": text

            })

print()

print("Total Pages:", len(documents))

Reading: SBI_Arbitrage_Opportunities_Fund.pdf
Reading: SBI_Corporate_Bond_Fund.pdf
Reading: SBI_Dividend_Yield_Fund.pdf
Reading: SBI_Fixed_Maturity_Plan_(FMP)-_Series_47_(1434_Days).pdf

Total Pages: 347


In [79]:
chunks = []

for doc in documents:

    chunks.append({
        "text": doc["text"],
        "source": doc["source"],
        "page": doc["page"]
    })

print("Total Chunks:", len(chunks))

Total Chunks: 347


In [48]:
chunks = []

for doc in documents:

    text_chunks = create_chunks(doc["text"])

    for chunk in text_chunks:

        chunks.append({

            "text": chunk,

            "source": doc["source"],

            "page": doc["page"]

        })

print("Total Chunks:", len(chunks))

Total Chunks: 3006


In [49]:
print(chunks[0]["source"])
print(chunks[0]["page"])
print()
print(chunks[0]["text"][:700])

SBI_Arbitrage_Opportunities_Fund.pdf
1

 
1   
 
SECTION I  
 
SCHEME INFORMATION DOCUMENT  
 
 
 
This product is suitable for 
investors  who are seeking*:  Scheme  Riskometer  Benchmark Riskometer  
 Short -term investment  
 Investments to exploit 
profitable arbitrage 
opportunities in the cash 
and derivative segments 
of the equity markets to 
provide capital 
appreciation and regular 
income.  
  
 As per AMFI Tier I  Benchmark  
 i.e. Nifty 50 Arbitrage Index  
 
 
*Investors should consult their financial advisers if in do


In [80]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded!


In [50]:
texts = []

for chunk in chunks:
    texts.append(chunk["text"])

print("Total Text Chunks:", len(texts))

Total Text Chunks: 3006


In [81]:
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print("Embeddings Shape:", embeddings.shape)

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Embeddings Shape: (3006, 384)


In [82]:
faiss.normalize_L2(embeddings)

print("Embeddings Normalized")

Embeddings Normalized


In [83]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS Index Created!")

print("Total Vectors:", index.ntotal)

FAISS Index Created!
Total Vectors: 3006


In [54]:
query = "What is the exit load?"

query_vector = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_vector)

distances, indices = index.search(
    query_vector,
    5
)

print(indices)

[[2281  137  138  136  894]]


In [73]:
for idx in indices[0]:

    print("=" * 80)

    print("Source :", chunks[idx]["source"])

    print("Page   :", chunks[idx]["page"])

    print()

    print(chunks[idx]["text"][:700])

Source : SBI_Fixed_Maturity_Plan_(FMP)-_Series_47_(1434_Days).pdf
Page   : 4

ange there will not be any Exit Load.    
 
Source : SBI_Arbitrage_Opportunities_Fund.pdf
Page   : 21

 limits 
specified in the SEBI regulations.  
 
 
D. LOAD STRUCTURE  
 
Exit Load is an amount which is paid by the investor to redeem the units from the scheme. Load amoun ts are 
variable and are subject to change from time to time. For the current applicable structure, please r efer to the 
website of the AMC (www.sbimf.com) or  may call at (toll free no. 1800 209 3333/1800 425 5425.) or your 
distributor.  
 
The following table illustrates the expenses that the investors will incur on their
Source : SBI_Arbitrage_Opportunities_Fund.pdf
Page   : 21

distributor.  
 
The following table illustrates the expenses that the investors will incur on their purchases/ sales  of Units during 
the continuous offer (including Systematic Investment Plan) under this scheme:  
 
Entry Load  Not Applicable  
Exit Load  

In [84]:
document_chunks = {}

for chunk in chunks:

    source = chunk["source"]

    if source not in document_chunks:
        document_chunks[source] = []

    document_chunks[source].append(chunk)

print("Documents Indexed:", len(document_chunks))

Documents Indexed: 4


In [85]:
document_indexes = {}

for source, doc_chunks in document_chunks.items():

    doc_texts = [chunk["text"] for chunk in doc_chunks]

    doc_embeddings = embedding_model.encode(
        doc_texts,
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(doc_embeddings)

    doc_index = faiss.IndexFlatIP(doc_embeddings.shape[1])

    doc_index.add(doc_embeddings)

    document_indexes[source] = {
        "index": doc_index,
        "chunks": doc_chunks
    }

print("Created", len(document_indexes), "document indexes.")

Created 4 document indexes.


In [86]:
for page in documents:

    if page["page"] == 21:

        print(page["source"])
        print(page["text"][:1500])

SBI_Arbitrage_Opportunities_Fund.pdf
 
21  the effective date of the change. Investors can refer https://www.sbimf.com/total -expense -ratio for Total 
Expense Ratio (TER) details.  
 
The additional TER in terms of Regulation 52(6A)(b) of SEBI (Mutual Funds) Regulations, 1996 shall b e 
charged based on inflows from Retail Investors from beyond top 30 cities (B-30 cities). Accordingly, the 
inflows of amount upto Rs 2,00,000/ - per transaction, by individual investors shall be considered as inflows 
from “Retail Investors  
 
Note:   SEBI vide its letter no.   SEBI/HO/IMD -SEC-3/P/OW/2023/5823/1 dated February 24, 2023 and AMFI 
letter dated No. 35P/ MEM -COR/ 85 -a/ 2022 -23 dated March 02, 2023 has directed AMCs to keep B -30 
incentive structure in  abeyance with effect from March 01, 2023 till further notice.  
 
 .Illustration of impact of expense ratio on schemes returns:  
 
   
Illustration of impact of expense ratio on scheme’s returns  
 Regular Plan  Direct Plan  
Opening 

In [77]:
for page in documents:

    if page["source"] == "SBI_Arbitrage_Opportunities_Fund.pdf" and page["page"] == 21:

        print(page["text"])

 
21  the effective date of the change. Investors can refer https://www.sbimf.com/total -expense -ratio for Total 
Expense Ratio (TER) details.  
 
The additional TER in terms of Regulation 52(6A)(b) of SEBI (Mutual Funds) Regulations, 1996 shall b e 
charged based on inflows from Retail Investors from beyond top 30 cities (B-30 cities). Accordingly, the 
inflows of amount upto Rs 2,00,000/ - per transaction, by individual investors shall be considered as inflows 
from “Retail Investors  
 
Note:   SEBI vide its letter no.   SEBI/HO/IMD -SEC-3/P/OW/2023/5823/1 dated February 24, 2023 and AMFI 
letter dated No. 35P/ MEM -COR/ 85 -a/ 2022 -23 dated March 02, 2023 has directed AMCs to keep B -30 
incentive structure in  abeyance with effect from March 01, 2023 till further notice.  
 
 .Illustration of impact of expense ratio on schemes returns:  
 
   
Illustration of impact of expense ratio on scheme’s returns  
 Regular Plan  Direct Plan  
Opening NAV (INR) (a)  100.00  100.00  
Schem

In [66]:
def retrieve(question, k=10):

    question_lower = question.lower()

    # Find matching document
    selected_doc = None

    for source in document_indexes.keys():

        filename = source.lower().replace(".pdf", "").replace("_", " ")

        if filename in question_lower:
            selected_doc = source
            break

    # Search only in selected document
    if selected_doc:

        search_chunks = document_chunks[selected_doc]

    else:

        search_chunks = chunks

    ##################################################
    # KEYWORD SEARCH FIRST
    ##################################################

    keyword_results = []

    keywords = question_lower.split()

    for chunk in search_chunks:

        score = 0

        text = chunk["text"].lower()

        for word in keywords:

            if word in text:
                score += 1

        if score > 0:

            keyword_results.append((score, chunk))

    keyword_results.sort(key=lambda x: x[0], reverse=True)

    if len(keyword_results) >= k:

        return [x[1] for x in keyword_results[:k]]

    ##################################################
    # FALLBACK TO FAISS
    ##################################################

    query = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query)

    if selected_doc:

        idx = document_indexes[selected_doc]["index"]

        doc_chunks = document_indexes[selected_doc]["chunks"]

        distances, indices = idx.search(query, k)

        return [doc_chunks[i] for i in indices[0]]

    distances, indices = index.search(query, k)

    return [chunks[i] for i in indices[0]]

In [87]:
results = retrieve(
    "What is the exit load of SBI Arbitrage Opportunities Fund?"
)

for r in results:

    print("="*80)
    print(r["page"])
    print(r["text"][:800])

3
 
3  Part I. HIGHLIGHTS/SUMMARY OF THE SCHEME  
 
Sr. No. Title Description  
I. Name  of the scheme  SBI Arbitrage Opportunities Fund  
II. Category  of the Scheme  Hybrid Schemes  - Arbitrage Fund  
 
III. Scheme  type An open ended scheme investing in arbitrage opportunities  
IV. Scheme  code  SBIM/O/H/ARB/06/03/0028  
V. Investment  objective  The investment objective of the Scheme is to provide capital 
appreciation and regular income for unitholders by identifying 
profitable arbitrage opportunities in the cash and derivative 
segments of the equity markets and the arbitrage opportunities 
available within the derivative segment and by investing the balance  
in debt and money market instruments.  
 
However there is no guarantee or assurance that the investment 
objective of the sc
4
 
4   
X. Plans  and Options  
Plans/Options  and sub 
options under the 
Scheme  The Scheme has Regular Plan and Direct plan.  Both plans have 
Growth & IDCW  options. IDCW option has Reinvestme

In [31]:
def build_context(results):

    context = ""

    for r in results:

        context += f"""
Source: {r['source']}
Page: {r['page']}

{r['text']}

"""

    return context

In [78]:
results = retrieve(
    "What is the exit load of SBI Arbitrage Opportunities Fund?"
)

for r in results:
    if r["page"] == 21:
        print(r["text"])

 limits 
specified in the SEBI regulations.  
 
 
D. LOAD STRUCTURE  
 
Exit Load is an amount which is paid by the investor to redeem the units from the scheme. Load amoun ts are 
variable and are subject to change from time to time. For the current applicable structure, please r efer to the 
website of the AMC (www.sbimf.com) or  may call at (toll free no. 1800 209 3333/1800 425 5425.) or your 
distributor.  
 
The following table illustrates the expenses that the investors will incur on their


In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen Loaded Successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen Loaded Successfully!


In [92]:
def generate_answer(question, context):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert assistant for SBI Mutual Fund documents. "
                "Answer ONLY using the provided context. "
                "If the answer is present, answer it directly in 2-4 sentences. "
                "Do not invent information."
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer

In [93]:
print(type(model))
print(model.config._name_or_path)

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Qwen/Qwen2.5-1.5B-Instruct


In [94]:
def ask_question(question):

    # Retrieve relevant chunks
    results = retrieve(question, k=2)

    if len(results) == 0:
        print("No relevant information found.")
        return

    # Build context
    context = ""

    for r in results:

        context += f"""
Source: {r['source']}
Page: {r['page']}

{r['text']}

"""

    # DEBUG: Print context being sent to LLM
    print("=" * 80)
    print("CONTEXT SENT TO LLM")
    print("=" * 80)
    print(context[:4000])      # print first 4000 characters
    print("=" * 80)

    # Generate answer
    answer = generate_answer(question, context)

    # Print answer
    print("\n" + "=" * 80)
    print("QUESTION")
    print(question)

    print("\nANSWER\n")
    print(answer)

    print("\nREFERENCES")

    shown = set()

    for r in results:

        ref = (r["source"], r["page"])

        if ref not in shown:
            print(f"{r['source']} | Page {r['page']}")
            shown.add(ref)

In [ ]:
while True:

    question = input("\nAsk a Question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    ask_question(question)


Ask a Question (type 'exit' to quit):  What is the exit load of SBI Arbitrage Opportunities Fund


CONTEXT SENT TO LLM

Source: SBI_Arbitrage_Opportunities_Fund.pdf
Page: 3

 
3  Part I. HIGHLIGHTS/SUMMARY OF THE SCHEME  
 
Sr. No. Title Description  
I. Name  of the scheme  SBI Arbitrage Opportunities Fund  
II. Category  of the Scheme  Hybrid Schemes  - Arbitrage Fund  
 
III. Scheme  type An open ended scheme investing in arbitrage opportunities  
IV. Scheme  code  SBIM/O/H/ARB/06/03/0028  
V. Investment  objective  The investment objective of the Scheme is to provide capital 
appreciation and regular income for unitholders by identifying 
profitable arbitrage opportunities in the cash and derivative 
segments of the equity markets and the arbitrage opportunities 
available within the derivative segment and by investing the balance  
in debt and money market instruments.  
 
However there is no guarantee or assurance that the investment 
objective of the scheme will be achieved. The scheme doesn’t 
assure or guarantee any returns.  
VI. Liquidity/listing  details  The scheme bein


Ask a Question (type 'exit' to quit):  what is dividend fund


CONTEXT SENT TO LLM

Source: SBI_Arbitrage_Opportunities_Fund.pdf
Page: 12

 
12  7. Floating rate debt instruments issued by central government, corporates, PSUs etc. with coupon re set 
periodically. The Fund Manager will have the flexibility to invest the debt component into floating rate debt 
securities in order to reduce the impact o f rising interest rate in the economy.  
8. Repo (Repurchase Agreement) or Reverse Repo  
9. Securitized Debt (SD)/Pass Through Certificate (PTC)  
10. Debt Derivatives  
11. Credit Default Swaps  
12. Repo transactions in corporate debt securities  
 
III. Other Securities  
1. Units of Mutual Fund  
2. Units issued by REITs/ InvITs  
3. Foreign Securities including ADR/GDR/Foreign equity and overseas ETFs and debt securities  
4. Instruments having special features  
 
Any other instruments / securities, which in the opinion of the fund manager would suit the investment 
objective/asset allocation of the scheme subject to compliance with extant Reg


Ask a Question (type 'exit' to quit):  What are the major risks associated with SBI Corporate Bond Fund?


CONTEXT SENT TO LLM

Source: SBI_Corporate_Bond_Fund.pdf
Page: 12

  
12   
c) Corporate debt securities to be bought by CDMDF during market dislocation include listed money market 
instruments. The long term rating of issuers shall be considered for the money market instruments. However, 
if there is no long term rating available for the same issuer, then based on credit rating mapping of CRAs 
between short term and long term ratings, the most conservative long term rating shall be taken for a given 
short term rating.  
 
d) CDMDF shall follow the Fair Pricing document, while purchase of corporate debt securities during market 
dislocation as specified in SEBI circular no. SEBI/HO/IMD/PoD2/P/CIR/2023/128 dated July 27, 2023 and 
circulars / guidelines/ Letters issued by SEBI and  AMFI from time to time  
 
e) CDMDF shall follow the loss waterfall accounting and guidelines w.r.t. purchase allocation and trade 
settlement of corporate debt securities bought by CDMDF, specified in chap


Ask a Question (type 'exit' to quit):  What securities can the fund invest in?


CONTEXT SENT TO LLM

Source: SBI_Arbitrage_Opportunities_Fund.pdf
Page: 12

 
12  7. Floating rate debt instruments issued by central government, corporates, PSUs etc. with coupon re set 
periodically. The Fund Manager will have the flexibility to invest the debt component into floating rate debt 
securities in order to reduce the impact o f rising interest rate in the economy.  
8. Repo (Repurchase Agreement) or Reverse Repo  
9. Securitized Debt (SD)/Pass Through Certificate (PTC)  
10. Debt Derivatives  
11. Credit Default Swaps  
12. Repo transactions in corporate debt securities  
 
III. Other Securities  
1. Units of Mutual Fund  
2. Units issued by REITs/ InvITs  
3. Foreign Securities including ADR/GDR/Foreign equity and overseas ETFs and debt securities  
4. Instruments having special features  
 
Any other instruments / securities, which in the opinion of the fund manager would suit the investment 
objective/asset allocation of the scheme subject to compliance with extant Reg